# IMDB：一次 Run All，顺序跑完三个官方 Trainer 实验

从公开预训练权重重新开始，依次训练 **BERT 全量 → BERT LoRA → DeBERTa-v2-xxlarge LoRA**。三个实验共用本次生成的分层划分，不读取此前的模型、训练日志或成绩，不需要中途切换模型变量。

训练采用 Hugging Face Transformers 的 `Trainer`、`TrainingArguments`，LoRA 使用官方 PEFT；数据集使用 Datasets，指标使用 scikit-learn，张量与 AMP 使用 PyTorch。Notebook 自身只负责读数据、配置实验和保存结果，不依赖仓库里的脚本或自定义训练框架。

## Kaggle 开始前

1. 新建/导入这份 Notebook，启动**新 Session**。
2. Settings 选择 **GPU T4 × 2**，打开 **Internet**。
3. Add Input 添加比赛 **Bag of Words Meets Bags of Popcorn**（`word2vec-nlp-tutorial`）。
4. 直接 **Run All**。默认全数据、每个模型 3 轮，不是抽样试跑。

需要的三个文件：

| 文件 | 用途 |
| --- | --- |
| `labeledTrainData.tsv.zip` 或解压后的 `labeledTrainData.tsv` | 25,000 条有标签影评，划为 20,000 训练 / 5,000 验证 |
| `testData.tsv.zip` 或解压后的 `testData.tsv` | 25,000 条比赛测试影评 |
| `sampleSubmission.csv` | 校验 ID，按官方顺序生成 CSV |

不需要无标签训练集、GloVe、旧代码、旧预测或额外模型数据集。公开模型在线下载，不强制提供 HF_TOKEN。

**设备说明：** 两张 T4 的显存不能直接合并。这份 Notebook 主动只向 Trainer 暴露第 1 张 GPU，三个模型逐个训练、预测和释放，避免默认数据并行复制 XXL。XXL 的默认长度 128、batch 1、梯度累积 16；真实运行的峰值显存会记录下来，尚不能保证任意 Kaggle 环境都不出现 OOM。XXL 下载和训练所需时间较长，请确保 Session 有足够时间。


In [ ]:
import os
import sys
from pathlib import Path
import csv
import zipfile

# 必须在首次 CUDA 初始化及首次导入 Hugging Face 库前执行。
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("CUDA 已初始化。请重启 Kaggle Session，然后从头 Run All。")
already_loaded = [name for name in ("transformers", "peft", "accelerate", "datasets") if name in sys.modules]
if already_loaded:
    raise RuntimeError(f"发现已加载的库 {already_loaded}。请重启 Session，再从头 Run All，避免安装后仍使用旧接口。")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

INPUT_ROOT = Path("/kaggle/input")
candidate_dirs = [
    INPUT_ROOT / "competitions/word2vec-nlp-tutorial",
    INPUT_ROOT / "word2vec-nlp-tutorial",
]
# 兼容比赛挂载目录变化，只接受同时包含这三个准确文件名的目录。
if INPUT_ROOT.exists():
    candidate_dirs += [p.parent for p in INPUT_ROOT.rglob("sampleSubmission.csv")]

def locate_inputs(directories):
    for directory in dict.fromkeys(directories):
        found = {}
        for key, stem in (
            ("train", "labeledTrainData.tsv"),
            ("test", "testData.tsv"),
            ("sample", "sampleSubmission.csv"),
        ):
            paths = [directory / stem]
            if stem.endswith(".tsv"):
                paths.append(directory / (stem + ".zip"))
            hit = next((p for p in paths if p.is_file()), None)
            if hit is not None:
                found[key] = hit
        if len(found) == 3:
            return found
    raise FileNotFoundError(
        "未找到比赛数据。请 Add Input 添加 word2vec-nlp-tutorial；"
        "必须同时包含 labeledTrainData.tsv(.zip)、testData.tsv(.zip)、sampleSubmission.csv。"
    )

INPUT_FILES = locate_inputs(candidate_dirs)
for key, path in INPUT_FILES.items():
    print(f"{key}: {path}")


## 安装官方依赖

保留 Kaggle 自带的 CUDA PyTorch、NumPy、pandas 和 scikit-learn。只固定 Hugging Face 核心库及 tokenizer 所需的 SentencePiece/Protobuf；不安装自定义库、bitsandbytes 或第三方训练器。以下版本的 Trainer 接口已核对。


In [ ]:
%pip install --quiet "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "sentencepiece==0.2.2" "protobuf>=4.25,<8"


In [ ]:
import gc
import html
import json
import math
import re
import time
import traceback
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, cast_mixed_precision_params
from IPython.display import display

VERSIONS = {name: version(name) for name in (
    "torch", "transformers", "peft", "accelerate", "datasets",
    "numpy", "pandas", "scikit-learn", "sentencepiece", "protobuf",
)}
print(json.dumps(VERSIONS, indent=2, ensure_ascii=False))
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA GPU。请在 Kaggle Settings 启用 T4 × 2 并重新启动 Session。")
if torch.cuda.device_count() != 1:
    raise RuntimeError("应只看到一张逻辑 GPU。请重启 Session，确认第一段代码首先执行。")
if tuple(int(p) for p in torch.__version__.split("+")[0].split(".")[:2]) < (2, 6):
    raise RuntimeError("需要 Kaggle CUDA PyTorch >= 2.6；请使用更新后的 Kaggle GPU 环境。")
print("本次设备:", torch.cuda.get_device_name(0))
print("可见 GPU 数:", torch.cuda.device_count())


## 读数据并固定本次划分

直接读取原始 TSV 或 ZIP，不需要手动解压。只去掉 HTML，保留大小写和标点，由各模型的官方 tokenizer 处理。训练/验证按标签分层、种子 42，三组使用完全相同的 ID。划分文件和运行版本会一起保存。


In [ ]:
SEED = 42
EPOCHS = 3
RUN_ROOT = Path("/kaggle/working") / (
    "imdb_official_three_models_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
RUN_ROOT.mkdir(parents=True, exist_ok=False)

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

def read_table(path, is_tsv=False):
    kwargs = {"dtype": {"id": str}}
    if is_tsv:
        kwargs.update(sep="\t", quoting=csv.QUOTE_NONE)
    path = Path(path)
    if path.suffix == ".zip":
        expected = path.name[:-4]
        with zipfile.ZipFile(path) as archive:
            members = [n for n in archive.namelist() if Path(n).name == expected and not n.endswith("/")]
            if len(members) != 1:
                raise ValueError(f"{path.name} 内应恰好有一个 {expected}，实际找到 {members}")
            with archive.open(members[0]) as stream:
                return pd.read_csv(stream, **kwargs)
    return pd.read_csv(path, **kwargs)

def normalized_ids(values):
    if values.isna().any():
        raise ValueError("ID 存在缺失值。")
    return values.astype(str).str.strip().str.strip('"').str.strip()

def clean_review(text):
    text = str(text).strip()
    if len(text) >= 2 and text.startswith('"') and text.endswith('"'):
        text = text[1:-1]  # TSV 的外层引号，不属于影评正文。
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]*>", " ", text))).strip()

def validate_frames(train, test, sample, expected_rows=25000):
    for name, frame, required in (
        ("train", train, {"id", "review", "sentiment"}),
        ("test", test, {"id", "review"}),
        ("sample", sample, {"id", "sentiment"}),
    ):
        if not required.issubset(frame.columns):
            raise ValueError(f"{name} 缺少列 {required - set(frame.columns)}")
        if len(frame) != expected_rows:
            raise ValueError(f"{name} 应有 {expected_rows} 条数据，实际 {len(frame)}")
        frame["id"] = normalized_ids(frame["id"])
        if frame["id"].eq("").any() or frame["id"].duplicated().any():
            raise ValueError(f"{name} 的 ID 为空或重复。")
    for name, frame in (("train", train), ("test", test)):
        if frame["review"].isna().any():
            raise ValueError(f"{name} 存在缺失影评。")
        frame["review"] = frame["review"].map(clean_review)
        if frame["review"].eq("").any():
            raise ValueError(f"{name} 清洗后存在空影评。")
    labels = pd.to_numeric(train["sentiment"], errors="raise")
    if set(labels.unique()) != {0, 1}:
        raise ValueError("训练标签必须是 0/1，且同时包含两类。")
    train["sentiment"] = labels.astype("int64")
    if set(test["id"]) != set(sample["id"]):
        raise ValueError("测试集与 sampleSubmission 的 ID 集合不一致。")
    # 训练/测试的原始影评文件名并非跨 split 的全局 ID；
    # 分别校验唯一性，禁止把两个不同数据文件仅按同名 ID 合并。
    return train, test, sample

train_all, test_frame, sample_frame = validate_frames(
    read_table(INPUT_FILES["train"], is_tsv=True),
    read_table(INPUT_FILES["test"], is_tsv=True),
    read_table(INPUT_FILES["sample"]),
)
train_indices, validation_indices = train_test_split(
    np.arange(len(train_all)), test_size=0.2,
    random_state=SEED, stratify=train_all["sentiment"],
)
train_frame = train_all.iloc[train_indices].reset_index(drop=True)
validation_frame = train_all.iloc[validation_indices].reset_index(drop=True)
split = pd.DataFrame({
    "source_row": np.concatenate([train_indices, validation_indices]),
    "id": pd.concat([train_frame["id"], validation_frame["id"]], ignore_index=True),
    "split": ["train"] * len(train_frame) + ["validation"] * len(validation_frame),
})
split.to_csv(RUN_ROOT / "split_ids.csv", index=False)
save_json(RUN_ROOT / "environment.json", {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "versions": VERSIONS, "seed": SEED,
    "input_files": {key: str(path) for key, path in INPUT_FILES.items()},
    "train_rows": len(train_frame), "validation_rows": len(validation_frame),
    "test_rows": len(test_frame),
    "gpu": torch.cuda.get_device_name(0), "visible_gpu_count": torch.cuda.device_count(),
    "fresh_pretrained_initialization": True,
})
print(f"训练 {len(train_frame)} / 验证 {len(validation_frame)} / 测试 {len(test_frame)}")
print("输出目录:", RUN_ROOT)
display(train_frame["sentiment"].value_counts().sort_index().rename("train_label_count"))


## 三个实验的默认配置

| 实验 | 预训练底座 | 训练方式 | 长度 | micro batch | 累积 | 有效 batch |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| bert_full | bert-base-uncased | 全量微调 | 256 | 8 | 2 | 16 |
| bert_lora | bert-base-uncased | LoRA + classifier | 256 | 8 | 2 | 16 |
| deberta_v2_xxlarge_lora | microsoft/deberta-v2-xxlarge | LoRA + pooler/classifier | 128 | 1 | 16 | 16 |

均训练 3 轮，AdamW、学习率 `2e-5`、weight decay `0.01`、线性调度、前 10% 步数 warmup；每轮验证并保存，按**验证 AUC**选择最佳检查点。LoRA：`r=8`、`alpha=16`、`dropout=0.05`。

BERT 全量与 BERT LoRA 是同底座、同长度的对照；XXL 的底座及截断长度都不同，比较时须保留这两项条件。这里没有添加已完成的旧成绩。


In [ ]:
EXPERIMENTS = [
    {
        "name": "bert_full", "checkpoint": "bert-base-uncased",
        "revision": "86b5e0934494bd15c9632b12f734a8a67f723594", "mode": "full",
        "max_length": 256, "train_batch_size": 8, "eval_batch_size": 16,
        "gradient_accumulation_steps": 2, "gradient_checkpointing": False,
        "frozen_backbone_fp16": False,
    },
    {
        "name": "bert_lora", "checkpoint": "bert-base-uncased",
        "revision": "86b5e0934494bd15c9632b12f734a8a67f723594", "mode": "lora",
        "max_length": 256, "train_batch_size": 8, "eval_batch_size": 16,
        "gradient_accumulation_steps": 2, "gradient_checkpointing": False,
        "frozen_backbone_fp16": False,
        "target_modules": ["query", "value"], "modules_to_save": ["classifier"],
    },
    {
        "name": "deberta_v2_xxlarge_lora", "checkpoint": "microsoft/deberta-v2-xxlarge",
        "revision": "38f115b79f551ca4091c9caef41940fd1d6312b9", "mode": "lora",
        "max_length": 128, "train_batch_size": 1, "eval_batch_size": 1,
        "gradient_accumulation_steps": 16, "gradient_checkpointing": True,
        "frozen_backbone_fp16": True,
        "target_modules": ["query_proj", "value_proj"],
        "modules_to_save": ["pooler", "classifier"],
    },
]
save_json(RUN_ROOT / "experiments.json", EXPERIMENTS)
display(pd.DataFrame(EXPERIMENTS)[[
    "name", "checkpoint", "mode", "max_length",
    "train_batch_size", "gradient_accumulation_steps",
]])


## 官方 Trainer 的准备与输出函数

模型类、损失、反向传播、优化器、混合精度和梯度累积均由官方模型与 Trainer 实现。下方函数负责 tokenizer、指标和 CSV 导出。验证准确率和 AUC 均按完整验证集计算；Trainer 默认不逐轮统计训练准确率，因此不会生成虚构的 train accuracy。

提交文件 `submission.csv` 的 `sentiment` 保存正类概率，供比赛 ROC-AUC 排序；`test_labels.csv` 另存阈值 0.5 的 0/1 预测。两份都使用提交模板的 ID 顺序。


In [ ]:
def tokenize_frame(frame, tokenizer, max_length, labeled):
    columns = ["review", "sentiment"] if labeled else ["review"]
    source = frame[columns].copy()
    if labeled:
        source = source.rename(columns={"sentiment": "labels"})
    dataset = Dataset.from_pandas(source, preserve_index=False)
    return dataset.map(
        lambda batch: tokenizer(batch["review"], truncation=True, max_length=max_length),
        batched=True, remove_columns=["review"],
        desc=f"Tokenize {len(frame)} rows / max_length={max_length}",
    )

def positive_probabilities(predictions):
    logits = predictions[0] if isinstance(predictions, tuple) else predictions
    logits = np.asarray(logits)
    if logits.ndim != 2 or logits.shape[1] != 2:
        raise ValueError(f"期望二分类 logits，实际 {logits.shape}")
    if not np.isfinite(logits).all():
        raise ValueError("模型输出存在 NaN/Inf，不能导出提交。")
    return torch.softmax(torch.from_numpy(logits).float(), dim=-1).numpy()[:, 1]

def compute_metrics(prediction):
    probabilities = positive_probabilities(prediction.predictions)
    labels = np.asarray(prediction.label_ids)
    return {
        "accuracy": float(accuracy_score(labels, (probabilities >= 0.5).astype("int64"))),
        "auc": float(roc_auc_score(labels, probabilities)),
    }

def ordered_submission(test_ids, template, probabilities):
    if len(test_ids) != len(probabilities):
        raise ValueError("预测数量与测试集数量不一致。")
    prediction = pd.Series(np.asarray(probabilities), index=normalized_ids(test_ids))
    if prediction.index.duplicated().any() or set(prediction.index) != set(template["id"]):
        raise ValueError("预测 ID 与提交模板不匹配。")
    result = template[["id"]].copy()
    result["sentiment"] = result["id"].map(prediction)
    if result["sentiment"].isna().any() or not result["sentiment"].between(0, 1).all():
        raise ValueError("提交概率存在缺失或超出 [0,1]。")
    return result

def make_training_args(spec, output_dir, train_rows, epochs=EPOCHS, use_fp16=True):
    updates_per_epoch = math.ceil(
        math.ceil(train_rows / spec["train_batch_size"]) / spec["gradient_accumulation_steps"]
    )
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=spec["train_batch_size"],
        per_device_eval_batch_size=spec["eval_batch_size"],
        gradient_accumulation_steps=spec["gradient_accumulation_steps"],
        learning_rate=2e-5, weight_decay=0.01,
        optim="adamw_torch", lr_scheduler_type="linear",
        warmup_steps=int(updates_per_epoch * epochs * 0.1),
        eval_strategy="epoch", save_strategy="epoch",
        save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model="auc", greater_is_better=True,
        fp16=use_fp16, bf16=False, fp16_full_eval=False,
        use_cpu=not torch.cuda.is_available(),
        gradient_checkpointing=spec["gradient_checkpointing"],
        gradient_checkpointing_kwargs={"use_reentrant": False},
        seed=SEED, data_seed=SEED,
        dataloader_num_workers=0,
        eval_accumulation_steps=16,
        label_names=["labels"], remove_unused_columns=True,
        logging_steps=100, report_to="none", push_to_hub=False,
    )


## 单次实验：训练、恢复最佳模型、保存与预测

XXL 以 FP16 载入底座，再由官方 PEFT 将可训练参数保持为 FP32；开启 AMP 和梯度检查点。BERT 两组使用 FP32 参数 + AMP。

每组从原始 checkpoint 初始化独立模型，`trainer.train(resume_from_checkpoint=None)` 明确不续接旧训练。训练结束时 Trainer 已恢复最佳 AUC 的模型，随后导出验证与测试预测。LoRA 的 `best_model` 保存 adapter 和任务头，重新载入还需要原始底座；BERT 全量保存完整模型。


In [ ]:
def run_experiment(spec, train, validation, test, template, run_root, epochs=EPOCHS, use_fp16=True):
    name = spec["name"]
    output_dir = Path(run_root) / name
    output_dir.mkdir(parents=True, exist_ok=False)
    trainer = model = tokenizer = None
    started = time.perf_counter()
    set_seed(SEED)  # 每组都在模型初始化前重新设定种子。
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    save_json(output_dir / "experiment.json", {**spec, "epochs": epochs, "seed": SEED})
    try:
        tokenizer = AutoTokenizer.from_pretrained(spec["checkpoint"], revision=spec["revision"])
        half_backbone = spec["frozen_backbone_fp16"] and use_fp16
        model = AutoModelForSequenceClassification.from_pretrained(
            spec["checkpoint"], revision=spec["revision"], num_labels=2,
            dtype=torch.float16 if half_backbone else torch.float32,
        )
        resolved_revision = getattr(model.config, "_commit_hash", None)
        if spec["mode"] == "lora":
            lora_config = LoraConfig(
                task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.05,
                bias="none", target_modules=spec["target_modules"],
                modules_to_save=spec["modules_to_save"],
                revision=resolved_revision or spec["revision"],
            )
            model = get_peft_model(model, lora_config)
            if half_backbone:
                cast_mixed_precision_params(model, dtype=torch.float16)
            model.print_trainable_parameters()
        elif spec["mode"] != "full":
            raise ValueError(f"未知训练方式: {spec['mode']}")
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        train_dataset = tokenize_frame(train, tokenizer, spec["max_length"], labeled=True)
        validation_dataset = tokenize_frame(validation, tokenizer, spec["max_length"], labeled=True)
        test_dataset = tokenize_frame(test, tokenizer, spec["max_length"], labeled=False)
        args = make_training_args(spec, output_dir / "checkpoints", len(train), epochs, use_fp16)
        (output_dir / "training_arguments.json").write_text(args.to_json_string() + "\n", encoding="utf-8")
        trainer = Trainer(
            model=model, args=args,
            train_dataset=train_dataset, eval_dataset=validation_dataset,
            processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
            compute_metrics=compute_metrics,
        )
        training_result = trainer.train(resume_from_checkpoint=None)
        history = list(trainer.state.log_history)
        best_checkpoint = trainer.state.best_model_checkpoint
        if best_checkpoint is None:
            raise RuntimeError("没有产生最佳验证检查点，不能声称训练已完成。")
        best_step = int(Path(best_checkpoint).name.rsplit("-", 1)[1])
        best_epoch = next(
            (row["epoch"] for row in history if row.get("step") == best_step and "eval_auc" in row),
            None,
        )
        save_json(output_dir / "log_history.json", history)
        pd.DataFrame(history).to_csv(output_dir / "training_log.csv", index=False)
        epoch_rows = [row for row in history if "eval_auc" in row]
        pd.DataFrame(epoch_rows).to_csv(output_dir / "epoch_validation.csv", index=False)
        save_json(output_dir / "train_metrics.json", training_result.metrics)
        trainer.state.save_to_json(str(output_dir / "trainer_state.json"))
        trainer.save_model(str(output_dir / "best_model"))
        peak_train_allocated = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else None
        peak_train_reserved = torch.cuda.max_memory_reserved() / 2**30 if torch.cuda.is_available() else None

        # 一次完整验证同时获取指标和逐样本概率，使用已恢复的最佳模型。
        validation_output = trainer.predict(validation_dataset, metric_key_prefix="validation")
        validation_probabilities = positive_probabilities(validation_output.predictions)
        pd.DataFrame({
            "id": validation["id"], "label": validation["sentiment"],
            "positive_probability": validation_probabilities,
            "prediction": (validation_probabilities >= 0.5).astype("int64"),
        }).to_csv(output_dir / "validation_predictions.csv", index=False)
        save_json(output_dir / "validation_metrics.json", validation_output.metrics)

        test_output = trainer.predict(test_dataset, metric_key_prefix="test")
        probabilities = positive_probabilities(test_output.predictions)
        submission = ordered_submission(test["id"], template, probabilities)
        submission.to_csv(output_dir / "submission.csv", index=False, float_format="%.9f")
        labels = submission.copy()
        labels["sentiment"] = (labels["sentiment"] >= 0.5).astype("int64")
        labels.to_csv(output_dir / "test_labels.csv", index=False)

        result = {
            "experiment": name, "status": "completed",
            "checkpoint": spec["checkpoint"], "resolved_revision": resolved_revision,
            "mode": spec["mode"], "max_length": spec["max_length"],
            "micro_batch": spec["train_batch_size"],
            "gradient_accumulation_steps": spec["gradient_accumulation_steps"],
            "effective_batch": spec["train_batch_size"] * spec["gradient_accumulation_steps"],
            "epochs": epochs, "best_epoch": best_epoch,
            "best_validation_auc_logged": float(trainer.state.best_metric),
            "validation_accuracy": validation_output.metrics["validation_accuracy"],
            "validation_auc": validation_output.metrics["validation_auc"],
            "validation_loss": validation_output.metrics["validation_loss"],
            "trainable_params": trainable_params, "total_params": total_params,
            "trainable_ratio": trainable_params / total_params,
            "trainer_global_steps": trainer.state.global_step,
            "warmup_steps": args.warmup_steps,
            "peak_train_allocated_gib": peak_train_allocated,
            "peak_train_reserved_gib": peak_train_reserved,
            "elapsed_seconds": time.perf_counter() - started,
            "test_rows": len(submission), "kaggle_score": None,
            "best_checkpoint": best_checkpoint,
            "best_model_dir": str(output_dir / "best_model"),
            "submission": str(output_dir / "submission.csv"),
        }
        save_json(output_dir / "result.json", result)
        print(f"{name} 完成：best_epoch={best_epoch}, "
              f"accuracy={result['validation_accuracy']:.4f}, AUC={result['validation_auc']:.6f}")
        return result
    finally:
        # Trainer / Accelerator / optimizer 都可能持有模型引用，必须一起释放。
        if trainer is not None:
            trainer.accelerator.free_memory()
        trainer = model = tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_all_experiments(specs, train, validation, test, template, run_root, epochs=EPOCHS, use_fp16=True):
    names = [spec["name"] for spec in specs]
    if not names or len(names) != len(set(names)):
        raise ValueError("实验列表须非空，且每组名称唯一。")
    existing = [name for name in names if (Path(run_root) / name).exists()]
    if existing:
        raise FileExistsError(
            f"当前输出目录已有实验 {existing}，为保护结果已停止。"
            "请重启 Session，再从头 Run All，创建新的本次目录。"
        )
    results = []
    for index, spec in enumerate(specs, start=1):
        print(f"\n{'=' * 24} [{index}/{len(specs)}] {spec['name']} {'=' * 24}")
        try:
            result = run_experiment(
                spec, train, validation, test, template, run_root, epochs, use_fp16,
            )
        except Exception as exc:
            error_dir = Path(run_root) / spec["name"]
            error_dir.mkdir(parents=True, exist_ok=True)
            (error_dir / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")
            result = {
                "experiment": spec["name"], "status": "failed",
                "checkpoint": spec["checkpoint"], "mode": spec["mode"],
                "max_length": spec["max_length"], "error": str(exc),
            }
            save_json(error_dir / "result.json", result)
            print(f"{spec['name']} 失败：{exc}。已保存 error.txt，继续下一组。")
        results.append(result)
        pd.DataFrame(results).to_csv(Path(run_root) / "results.csv", index=False)
        save_json(Path(run_root) / "results.json", results)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(results)


## Run All 的主入口

下方自动遍历全部三组，不需要编辑模型变量。每组结束就更新总表；若某组失败，记录错误并继续其余组，最后明确列出失败项，不会把部分成功写成“三个都完成”。


In [ ]:
results = run_all_experiments(
    EXPERIMENTS, train_frame, validation_frame, test_frame, sample_frame, RUN_ROOT,
)
display(results)
failed = results.loc[results["status"] != "completed", "experiment"].tolist()
if failed:
    raise RuntimeError(f"以下实验未完成：{failed}。请查看各组 error.txt；已完成的结果保存在 {RUN_ROOT}")
print("三个实验均已完成。输出目录:", RUN_ROOT)
for row in results.to_dict("records"):
    print(row["experiment"], "→", row["submission"])


## 跑完后看什么、下载什么

- `results.csv`：本次三组验证 accuracy/AUC、最佳轮次、训练参数、输入长度、显存、运行时间、提交路径。这里的验证 AUC 不是 Kaggle 排行榜成绩。
- `split_ids.csv`、`environment.json`、`experiments.json`：本次划分、真实库版本与模型条件。
- 每组的 `epoch_validation.csv` / `training_log.csv`：每轮验证与 Trainer 原始日志。
- 每组的 `best_model/`：最佳模型或 LoRA adapter + 任务头；tokenizer 一起保存。
- 每组的 `validation_predictions.csv`：5,000 条真实验证标签、概率和预测，可检查误分类。
- 每组的 `submission.csv`：25,000 条比赛测试正类概率，可提交 Kaggle；`test_labels.csv` 是另存的 0/1 版本。

在 Kaggle Output 下载本次 `imdb_official_three_models_...` 目录。三份 `submission.csv` 分别属于三个实验，请保留目录名。Notebook 不自动向 Kaggle 提交；实际分数以提交后的页面为准。

若 XXL 失败，错误和前两组结果仍在本次目录中。日志中的分类头 MISSING / 预训练任务头 UNEXPECTED 通常表示加载的是预训练底座、分类头需在本任务训练。匿名 Hugging Face 下载的限速提示也不等于训练失败；以异常和最终 status 为准。

### 官方依据

- [Transformers Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer)
- [固定版本 TrainingArguments 实现](https://github.com/huggingface/transformers/blob/v5.19.0/src/transformers/training_args.py)
- [PEFT LoRA](https://huggingface.co/docs/peft/en/package_reference/lora)
- [PEFT 混合参数精度处理](https://github.com/huggingface/peft/blob/v0.21.2/src/peft/utils/other.py)
- [DeBERTa-v2-xxlarge 模型卡](https://huggingface.co/microsoft/deberta-v2-xxlarge)
- [PyTorch 梯度检查点](https://docs.pytorch.org/docs/stable/checkpoint.html)
- [比赛数据](https://www.kaggle.com/competitions/word2vec-nlp-tutorial/data)

本文件交付时未写入训练输出。真实 T4 / XXL 的训练指标和显存必须由本次 Kaggle 执行产生。
